In [ ]:
!pip -q install transformers sentencepiece sacrebleu openpyxl pandas accelerate

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import pandas as pd

df = pd.read_excel("english_hindi_dataset_analysis.xlsx")

df.head()

In [ ]:
sample = df.iloc[:100].copy()

sample.head()

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"

model.to(device)

print(device)

In [ ]:
translations = []

for sentence in sample["english"]:

    inputs = tokenizer(
        sentence,
        return_tensors="pt"
    ).to(device)

    translated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids("hin_Deva"),
        max_length=256
    )

    hindi = tokenizer.batch_decode(
        translated_tokens,
        skip_special_tokens=True
    )[0]

    translations.append(hindi)

sample["Generated Hindi"] = translations

sample.head()

In [ ]:
from sacrebleu.metrics import BLEU, CHRF, TER

bleu = BLEU()
chrf = CHRF()
ter = TER()

references = [[x] for x in sample["hindi"]]
hypothesis = sample["Generated Hindi"].tolist()

bleu_score = bleu.corpus_score(hypothesis, references)

chrf_score = chrf.corpus_score(hypothesis, references)

ter_score = ter.corpus_score(hypothesis, references)

print("BLEU :", bleu_score.score)
print("CHRF :", chrf_score.score)
print("TER :", ter_score.score)

In [ ]:
sample.to_excel(
    "Assignment2_Translations.xlsx",
    index=False
)